In [13]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, KFold
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)

from xgboost import XGBRegressor
import joblib

In [14]:
candidates = [
        (
            "baseline",
            DummyRegressor(strategy="median"),
            {},
        ),
        (
            "hist_gradient_boosting",
            HistGradientBoostingRegressor(
                random_state=42,
                early_stopping=True,
                max_iter=500,
                validation_fraction=0.15,
                scoring="neg_mean_absolute_error",
                n_iter_no_change=10,
            ),
            {
                "learning_rate": [0.05, 0.1],
                "max_leaf_nodes": [15, 31],
                "min_samples_leaf": [5, 20],
                "l2_regularization": [0.0, 1.0],
            },
        ),
        (
            "random_forest",
            RandomForestRegressor(
                random_state=42,
                n_jobs=1,
            ),
            {
                "n_estimators": [200],
                "max_depth": [None, 15],
                "min_samples_leaf": [1, 3],
                "max_features": [0.5, 1.0],
            },
        ),
        (
            "extra_trees",
            ExtraTreesRegressor(
                random_state=42,
                n_jobs=1,
            ),
            {
                "n_estimators": [200],
                "max_depth": [None, 15],
                "min_samples_leaf": [1, 3],
                "max_features": [0.5, 1.0],
            },
        ),
        (
            "xgboost",
            XGBRegressor(
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=1,
            ),
            {
                "n_estimators": [200],
                "learning_rate": [0.05, 0.1],
                "max_depth": [3, 6],
                "min_child_weight": [1, 5],
                "subsample": [0.8],
                "colsample_bytree": [0.8],
                "reg_lambda": [1.0],
            },
        ),
    ]

def train_models():
    global candidates
    
    models = {}
    results = []
    
    cv = KFold(
        n_splits=3,
        shuffle=True,
        random_state=42,
    )
    
    X_train = pd.read_json("./train_data/X_train.json")
    X_test = pd.read_json("./train_data/X_test.json")
    y_train = pd.read_json("./train_data/y_train.json")
    y_test = pd.read_json("./train_data/y_test.json")
    
    models = {}
    results = []
    
    # predict a single column at a time across input features
    for target in y_train.columns:
        available = y_train[target].notna()
        target_X = X_train.loc[available]
        target_y = y_train.loc[available, target]
        
        if target_y.empty:
            print(f"{target}: skipped — no observed training values")
            continue
        
        best_model = None
        best_name = None
        best_params = {}
        best_cv_score = None
        
        for name, estimator, parameters in candidates:
            search = GridSearchCV(
                estimator=estimator, 
                param_grid=parameters, 
                scoring="neg_mean_absolute_error",
                cv=cv, 
                n_jobs=-1, 
                error_score="raise",
                verbose=2,
            )
            
            search.fit(target_X, target_y)

            print(
                f"{target} | {name} | "
                f"CV MAE: {-search.best_score_:.4f}",
                flush=True,
            )
            
            if best_cv_score is None or search.best_score_ > best_cv_score:
                best_model = search.best_estimator_
                best_name = name
                best_params = search.best_params_
                best_cv_score = search.best_score_
        models[target] = best_model
        
        available_test = y_test[target].notna()
        test_mae = None
        
        if available_test.any():
            test_predictions = best_model.predict(
                X_test.loc[available_test]
            )

            test_mae = mean_absolute_error(
                y_test.loc[available_test, target],
                test_predictions,
            )

        cv_mae = None
        if best_cv_score is not None:
            cv_mae = -best_cv_score

        results.append({
            "target": target,
            "model": best_name,
            "parameters": str(best_params),
            "cv_mae": cv_mae,
            "test_mae": test_mae,
        })

        print(
            f"Selected {best_name} for {target} | "
            f"Test MAE: {test_mae}",
            flush=True,
        )
    
    joblib.dump(models, "./model/models.joblib")
    
    results = pd.DataFrame(results)
    results.to_csv("./modelresults.csv", index=False)

In [15]:
import time

def benchmark_models():
    models = joblib.load("./model/models.joblib")
    
    X_train = pd.read_json("./train_data/X_train.json")
    X_test = pd.read_json("./train_data/X_test.json")
    
    X = pd.concat([X_train, X_test])


    start = time.perf_counter()
    predicted_changes = pd.DataFrame(index=X.index)
    for target, model in models.items():
        predicted_changes[target] = model.predict(X)

    elapsed = time.perf_counter() - start

    print(f"Units: {len(X)}")
    print(f"Models: {len(models)}")
    print(f"Total prediction time: {elapsed:.4f} seconds")

    if len(X) > 0:
        print(f"Time per unit: {elapsed / len(X) * 1000:.4f} ms")
benchmark_models()

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- numeric__is_ranged
Feature names seen at fit time, yet now missing:
- numeric__missingindicator_projectiles__ap_damage
- numeric__missingindicator_projectiles__base_reload_time
- numeric__missingindicator_projectiles__bonus_v_infantry
- numeric__missingindicator_projectiles__bonus_v_large
- numeric__missingindicator_projectiles__damage
- ...


In [ ]:
models = joblib.load("./model/models.joblib")
preprocessor = joblib.load("./train_data/preprocessor.joblib")
vanilla = pd.read_json("./cleaned_data/vanilla.json")

def predict_unit(unit_key):
    unit = vanilla.loc[vanilla["unit"] == unit_key]

    if unit.empty:
        raise ValueError(f"Unit not found: {unit_key}")

    unit = unit.iloc[[0]]
    unit["is_ranged"] = (
        unit["land_units__primary_missile_weapon"]
        .notna()
        .astype(int)
    )
    inputs = unit.loc[:, preprocessor.feature_names_in_]

    processed = pd.DataFrame(
        preprocessor.transform(inputs),
        columns=preprocessor.get_feature_names_out(),
        index=inputs.index,
    )

    changes = {}

    for target, model in models.items():
        stat = target.removeprefix("delta_")
        changes[stat] = float(model.predict(processed)[0])

    return changes
print(predict_unit("wh2_dlc09_skv_cha_tretch_craventail"))

{'main_units__tier': 0.0, 'main_units__num_men': 0.0, 'main_units__recruitment_cost': 0.0, 'main_units__upkeep_cost': -125.0, 'main_units__multiplayer_cost': 200.0, 'main_units__barrier_health': 0.0, 'land_units__accuracy': 0.0, 'land_units__charge_bonus': 5.0, 'land_units__melee_attack': 0.0, 'land_units__melee_defence': 0.0, 'land_units__morale': 0.0, 'land_units__bonus_hit_points': 860.0, 'land_units__num_mounts': 0.0, 'land_units__secondary_ammo': 0.0, 'land_units__primary_ammo': 0.0, 'land_units__damage_mod_flame': 0.0, 'land_units__damage_mod_magic': 0.0, 'land_units__num_engines': 0.0, 'land_units__damage_mod_physical': 0.0, 'land_units__damage_mod_missile': 0.0, 'land_units__damage_mod_all': 0.0, 'land_units__reload': 0.0, 'battle_entities__run_speed': 0.0, 'battle_entities__charge_speed': 0.0, 'battle_entities__mass': 31.575, 'battle_entities__hit_points': 2.0, 'melee_weapons__bonus_v_large': 11.0, 'melee_weapons__bonus_v_infantry': 0.0, 'melee_weapons__damage': 50.0, 'melee_w

In [16]:
if __name__ == "__main__":
    train_models()

Fitting 3 folds for each of 1 candidates, totalling 3 fits
delta_main_units__tier | baseline | CV MAE: 0.3396
Fitting 3 folds for each of 16 candidates, totalling 48 fits
delta_main_units__tier | hist_gradient_boosting | CV MAE: 0.2166
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__tier | random_forest | CV MAE: 0.2164
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__tier | extra_trees | CV MAE: 0.2085
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__tier | xgboost | CV MAE: 0.2155
Selected extra_trees for delta_main_units__tier | Test MAE: 0.19617424242424242
Fitting 3 folds for each of 1 candidates, totalling 3 fits
delta_main_units__num_men | baseline | CV MAE: 0.7407
Fitting 3 folds for each of 16 candidates, totalling 48 fits
delta_main_units__num_men | hist_gradient_boosting | CV MAE: 0.9335
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__num_men | random_forest 

In [3]:
import json
from pathlib import Path
from copy import deepcopy
import joblib
from sklearn.dummy import DummyRegressor
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType
from xgboost import XGBRegressor
from skl2onnx import update_registered_converter
from skl2onnx.common.shape_calculator import (
    calculate_linear_regressor_output_shapes,
)

from onnxmltools.convert.xgboost.operator_converters.XGBoost import (
    convert_xgboost,
)

update_registered_converter(
    XGBRegressor,
    "XGBoostXGBRegressor",
    calculate_linear_regressor_output_shapes,
    convert_xgboost,
)

models = joblib.load("./model/models.joblib")

manifest = {}

for target, model in models.items():
    if isinstance(model, DummyRegressor):
        manifest[target] = {
            "constant": float(model.constant_.ravel()[0])
        }
        continue

    export_model = model

    if isinstance(model, XGBRegressor):
        export_model = deepcopy(model)

        feature_names = []

        for i in range(model.n_features_in_):
            feature_names.append(f"f{i}")

        export_model.get_booster().feature_names = feature_names

    onnx_model = to_onnx(
        export_model,
        initial_types=[
            ("features", FloatTensorType([None, model.n_features_in_]))
        ],
        target_opset={"": 17, "ai.onnx.ml": 3},
    )

    filename = f"{target}.onnx"

    with open(f"./onnx_models/{filename}", "wb") as file:
        file.write(onnx_model.SerializeToString())

    manifest[target] = {"file": filename}

with open(f"./onnx_models/models.json", "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

In [ ]:
import json
from copy import deepcopy

import joblib
from skl2onnx import to_onnx
from skl2onnx.common.data_types import (
    FloatTensorType,
    StringTensorType,
)



preprocessor = joblib.load("./train_data/preprocessor.joblib")

categorical_columns = []s

for name, transformer, columns in preprocessor.transformers_:
    if name == "categorical":
        categorical_columns = list(columns)

# ONNX string tensors cannot contain null.
# Use a string marker for missing categorical inputs.
missing_category = "__MISSING__"

export_preprocessor = deepcopy(preprocessor)
categorical_pipeline = export_preprocessor.named_transformers_["categorical"]
categorical_pipeline.named_steps["impute"].missing_values = missing_category

input_types = []
input_schema = []

for column in preprocessor.feature_names_in_:
    if column in categorical_columns:
        tensor_type = StringTensorType([None, 1])
        input_type = "string"
    else:
        tensor_type = FloatTensorType([None, 1])
        input_type = "float32"

    input_types.append((column, tensor_type))
    input_schema.append({
        "name": column,
        "type": input_type,
    })

preprocessor_onnx = to_onnx(
    export_preprocessor,
    initial_types=input_types,
    target_opset={"": 17, "ai.onnx.ml": 3},
)

with open("./onnx_models/preprocessor.onnx", "wb") as file:
    file.write(preprocessor_onnx.SerializeToString())

schema = {
    "inputs": input_schema,
    "categorical_missing": missing_category,
    "output": preprocessor_onnx.graph.output[0].name,
}

with open("./onnx_models/preprocessor.json", "w", encoding="utf-8",) as file:
    json.dump(schema, file, indent=2)